# Worked Examples — Intermediate

Six explained solutions covering classes, generators, decorators, JSON, SQL and regex. None of them are in the graded practice.

Each problem goes **plan → solution → why it works → common mistake → your turn**. Cover the solution, write your own version first, then compare.

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "worked_examples"))
    sys.path.insert(0, os.getcwd())

## 1. A loan with a computed late fee

Model a library loan with a due date. Its `late_fee` property is €0.50 per day late, capped at €10.

**Plan**

Use a dataclass for the data and a property for the derived value. Pass 'today' in as a parameter so the code is testable.

In [ ]:
from dataclasses import dataclass
from datetime import date

@dataclass
class Loan:
    title: str
    due: date

    def late_fee(self, today: date) -> float:
        days_late = max(0, (today - self.due).days)
        return min(10.0, days_late * 0.5)

loan = Loan("Dune", due=date(2024, 3, 1))
print(loan.late_fee(date(2024, 3, 5)))
assert loan.late_fee(date(2024, 2, 28)) == 0      # early
assert loan.late_fee(date(2024, 3, 5)) == 2.0
assert loan.late_fee(date(2024, 6, 1)) == 10.0    # capped

**Why it works:** `max(0, ...)` handles early returns and `min(10, ...)` applies the cap. Taking `today` as an argument keeps the tests deterministic.

**Common mistake:** Calling `date.today()` inside the method, which makes the tests depend on the day you run them.

**Your turn:** Add a `renew(days)` method that pushes the due date forward, at most twice.

## 2. Lazy log pipeline

From log lines like `'2024-01-01 ERROR db: timeout'`, count ERROR lines per module without loading the whole file into memory.

**Plan**

Build three small generator stages: parse → keep errors → count. Each line flows through one at a time.

In [ ]:
from collections import Counter

def parse(lines):
    for line in lines:
        parts = line.split(maxsplit=3)
        if len(parts) == 4:
            day, level, module, message = parts
            yield {"level": level, "module": module.rstrip(":"), "message": message}

def errors(records):
    return (r for r in records if r["level"] == "ERROR")

log = ["2024-01-01 ERROR db: timeout", "2024-01-01 INFO web: ok",
       "2024-01-02 ERROR db: refused", "garbage", "2024-01-02 ERROR web: 500"]
counts = Counter(r["module"] for r in errors(parse(log)))
print(counts)
assert counts == {"db": 2, "web": 1}

**Why it works:** The generators are lazy, so the same code works on a 10 GB file opened with `open()`. The malformed line is skipped by the length check.

**Common mistake:** Calling `list()` between the stages, which defeats laziness and uses memory for the whole file.

**Your turn:** Add a stage that keeps only lines from a given date.

## 3. A timing decorator with an injected clock

Write `@timed(clock)` that records how long each call took in `wrapper.timings`.

**Plan**

A decorator factory takes the clock. The wrapper reads the clock before and after, in `try/finally` so failed calls are timed too.

In [ ]:
import functools
import time

def timed(clock=time.perf_counter):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            start = clock()
            try:
                return func(*args, **kwargs)
            finally:
                wrapper.timings.append(clock() - start)
        wrapper.timings = []
        return wrapper
    return decorator

fake_now = iter([0.0, 1.5, 10.0, 10.25])
@timed(clock=lambda: next(fake_now))
def work(x):
    return x * 2

assert work(2) == 4 and work(3) == 6
print(work.timings)
assert work.timings == [1.5, 0.25]

**Why it works:** Injecting the clock makes the timings exact in tests. `finally` records the time even when `func` raises.

**Common mistake:** Forgetting `functools.wraps`, so `work.__name__` becomes `'wrapper'` in logs and tracebacks.

**Your turn:** Add a `slowest` attribute that stores the longest timing seen.

## 4. Validated order with a JSON round trip

An `Order` has items `(name, qty, unit_price)`. Reject non-positive quantities, compute the total, and save to and load from JSON.

**Plan**

A dataclass with `__post_init__` validation, a `total` property, and `to_json` / `from_json` that convert through plain dicts.

In [ ]:
import json
from dataclasses import dataclass, field, asdict

@dataclass
class Order:
    customer: str
    items: list[tuple[str, int, float]] = field(default_factory=list)

    def __post_init__(self):
        self.items = [tuple(item) for item in self.items]   # JSON gives lists back
        if any(qty <= 0 for _, qty, _ in self.items):
            raise ValueError("quantities must be positive")

    @property
    def total(self):
        return round(sum(qty * price for _, qty, price in self.items), 2)

    def to_json(self):
        return json.dumps(asdict(self))

    @classmethod
    def from_json(cls, text):
        return cls(**json.loads(text))

order = Order("ana", [("pen", 2, 1.25), ("ink", 1, 4.0)])
copy = Order.from_json(order.to_json())
print(copy, copy.total)
assert copy == order and copy.total == 6.5

**Why it works:** `asdict` produces plain data. `__post_init__` runs on every construction, including `from_json`, so loaded data is validated too.

**Common mistake:** Comparing a loaded order with the original when JSON has turned the tuples into lists. The normalization line fixes this.

**Your turn:** Reject unknown keys in `from_json` with a clear error.

## 5. Top products by revenue in SQL

With tables `products(id, name)` and `sales(product_id, qty, price)`, list the top 2 products by revenue.

**Plan**

JOIN the tables, GROUP BY product, ORDER BY the revenue sum, LIMIT with a parameter.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE sales (product_id INTEGER REFERENCES products(id), qty INTEGER, price REAL);
INSERT INTO products VALUES (1, 'pen'), (2, 'ink'), (3, 'pad');
INSERT INTO sales VALUES (1, 10, 1.5), (2, 2, 9.0), (3, 1, 3.0), (1, 4, 1.5), (2, 1, 9.0);
""")

def top_products(conn, limit):
    query = """
        SELECT p.name, ROUND(SUM(s.qty * s.price), 2) AS revenue
        FROM sales AS s JOIN products AS p ON p.id = s.product_id
        GROUP BY p.id
        ORDER BY revenue DESC
        LIMIT ?"""
    return conn.execute(query, (limit,)).fetchall()

print(top_products(conn, 2))
assert top_products(conn, 2) == [("ink", 27.0), ("pen", 21.0)]

**Why it works:** The database does the aggregation in one query. `LIMIT ?` keeps the query parameterized.

**Common mistake:** Grouping by `p.name`, which merges two different products that happen to share a name. Group by the id.

**Your turn:** Include products with no sales (revenue 0) using a LEFT JOIN and COALESCE.

## 6. Parse key=value pairs with a regex

Extract `user=ana status=200 path="/a b"` into a dict. Values may be quoted and contain spaces.

**Plan**

One pattern with two alternatives for the value: a quoted string or a run of non-space characters. Use named groups and `finditer`.

In [ ]:
import re

PAIR = re.compile(r'(?P<key>\w+)=(?:"(?P<quoted>[^"]*)"|(?P<plain>\S+))')

def parse_pairs(line):
    return {m["key"]: m["quoted"] if m["quoted"] is not None else m["plain"]
            for m in PAIR.finditer(line)}

line = 'user=ana status=200 path="/a b" empty=""'
print(parse_pairs(line))
assert parse_pairs(line) == {"user": "ana", "status": "200", "path": "/a b", "empty": ""}

**Why it works:** Quoted values are matched first, so spaces inside quotes are kept. Checking `is not None` keeps empty quoted values like `""`.

**Common mistake:** Writing `m['quoted'] or m['plain']`, which turns `empty=""` into `None` because `''` is falsy.

**Your turn:** Convert values that look like integers into `int`.

**Next:** the [Intermediate projects](../projects/) and [auto-graded practice](../../practice/README.md).